# Pure Text-Driven Video Generation Training (EchoMimic Adaptation)

This notebook helps you train a text-to-feature mapping model to drive EchoMimic without speech audio.

## 1. Environment Setup

In [ ]:
!cp drive/MyDrive/playground.zip playground.zip

In [2]:
%cd content
!git clone https://github.com/windaoist/Text2Lip.git

/content
Cloning into 'Text2Lip'...
remote: Enumerating objects: 447, done.
remote: Total 447 (delta 0), reused 0 (delta 0), pack-reused 447 (from 1)
Receiving objects: 100% (447/447), 74.26 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [39]:
%cd /content/playground
!git pull

/content/playground
Already up to date.


In [ ]:
import os

# 压缩 playground 文件夹，排除 pretrained_weights 和 data 目录
!zip -r playground_backup.zip /content/playground -x "/content/playground/pretrained_weights/*" "/content/playground/data/*"

# 将压缩包复制到 Google Drive
!cp playground_backup.zip /content/drive/MyDrive/

print("压缩并转移完成。文件已保存至 Drive 根目录下的 playground_backup.zip")

In [3]:
!pip install -r /content/playground/requirements.txt

Looking in indexes: https://mirrors.tencent.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 53.9 MB/s  0:00:00m0:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.8 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 38.9 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 51.6 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.2 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 9.9 MB/s  0:00:036m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.2 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 54.8 MB/s  0:00:00m0:00:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Pre

In [ ]:
# Mount Google Drive if needed
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
import nltk

# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev pkg-config

# Install python packages with versions compatible with Python 3.12
# Using torch>=2.2.0 as 2.1.2 is not available for 3.12
# Updated mediapipe to 0.10.13 as 0.10.9 is not found
!pip install -q "numpy<2.0.0" "torch>=2.2.0" "torchvision>=0.17.0" "torchaudio>=2.2.0" \
    g2p_en nltk opencv-python matplotlib diffusers==0.24.0 transformers==4.38.1 \
    accelerate omegaconf==2.3.0 einops==0.4.1 mediapipe>=0.10.13 moviepy==1.0.3 \
    facenet-pytorch==2.5.0 gradio

# Force NumPy 1.x again to ensure compatibility with EchoMimic modules
!pip install -q "numpy<2.0.0"

# Download required NLTK data to prevent training crashes
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print("Environment setup complete. PLEASE RESTART THE RUNTIME (Runtime -> Restart session) now to apply NumPy changes.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122402 files and directories currently installed.)
Removing r-base-dev (4.5.3-1.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libglib2.0-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libfontconfig-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.

Re

In [ ]:
import sys
# Standardize on ffmpeg-python which is the most common wrapper for EchoMimic
!pip uninstall -y python-ffmpeg ffmpeg
!pip install -q ffmpeg-python
print("ffmpeg-python installed and conflicting packages removed.")

ffmpeg-python installed and conflicting packages removed.


In [ ]:
!pip install -q av
print('PyAV (av) installed.')

PyAV (av) installed.


In [ ]:
import sys
# Ensure huggingface_hub is <= 0.25.0 as per EchoMimic requirements
# Also ensuring transformers is in a range compatible with this hub version
!pip install -q "huggingface_hub<=0.25.0" "transformers>=4.38.1,<4.41.0"
print("Dependencies adjusted. Please restart the runtime if you still see ImportErrors.")

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import cv2
import numpy as np
import nltk
from g2p_en import G2p
import matplotlib.pyplot as plt

# 确保下载 NLTK 依赖
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


In [5]:
import nltk
# 显式下载到默认路径以确保训练脚本能找到该资源
nltk.download('averaged_perceptron_tagger_eng', download_dir='/root/nltk_data')
print('NLTK resource downloaded.')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...


NLTK resource downloaded.


[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


## 2. Clone Repository & Setup EchoMimic

Assuming you have uploaded the project files to your Drive or you are cloning them.

In [ ]:

!unzip playground.zip -d /content/playground

Archive:  playground.zip
  inflating: /content/playground/Colab_training.ipynb  
   creating: /content/playground/configs/
  inflating: /content/playground/configs/animation.yaml  
  inflating: /content/playground/configs/environment.yml  
   creating: /content/playground/configs/inference/
  inflating: /content/playground/configs/inference/inference_v1.yaml  
  inflating: /content/playground/configs/inference/inference_v2.yaml  
   creating: /content/playground/configs/prompts/
  inflating: /content/playground/configs/prompts/animation.yaml  
  inflating: /content/playground/configs/prompts/animation_acc.yaml  
  inflating: /content/playground/configs/prompts/animation_pose.yaml  
  inflating: /content/playground/configs/prompts/animation_pose_acc.yaml  
  inflating: /content/playground/configs/text_driven_config.yaml  
   creating: /content/playground/data/
   creating: /content/playground/data/audio/
  inflating: /content/playground/data/audio/temp_prototype_audio.wav  
  inflating:

## 3. Data Preparation

Upload your video dataset (e.g., AVDigits) to `data/asserts/AVDigits`.

In [6]:
!python /content/playground/scripts/download_models.py

开始通过国内镜像源下载模型到: /content/playground/pretrained_weights...

[1/3] 正在下载 sd-vae-ft-mse...
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/huggingface_hub/file_download.py:1204: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 5 files:   0%|                                   | 0/5 [00:00<?, ?it/s]
README.md: 6.84kB [00:00, 10.9MB/s]

config.j

## 4. Start Training

Run the distillation training script to learn the text-to-feature mapping.

In [ ]:
import ffmpeg
# In ffmpeg-python, the error class is usually accessed via ffmpeg.Error
try:
   # This is just a check to see if the module is loaded correctly
   print(f"FFmpeg module location: {ffmpeg.__file__}")
   # If you need to catch ffmpeg errors, use ffmpeg.Error
   print("FFmpeg Error class is accessible.")
except AttributeError:
   print("FFmpeg Error class still not found. Try restarting the runtime.")

FFmpeg module location: /usr/local/lib/python3.12/dist-packages/ffmpeg/__init__.py
FFmpeg Error class is accessible.


### Download GRID Dataset (Subset s1)
Since the dataset files are large and missing from the zip, we will download the 's1' subset and its corresponding alignments.

In [7]:
%cd /content/
!curl https://spandh.dcs.shef.ac.uk//gridcorpus/s1/video/s1.mpg_vcd.zip --output s1.mpg_vcd.zip

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  2  403M    2 8429k    0     0  18527      0  6:20:17  0:07:45  6:12:32 1694545  7:15:30 136025  0:05:34  6:35:41 17795:45  0:07:35  6:14:10 21853^C


In [35]:
import os

# Define the target path
target_dir = "/content/playground/pretrained_weights"
os.makedirs(target_dir, exist_ok=True)

# Using the verified link from camenduru (converting blob to resolve for direct download)
download_url = "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/lipsync_expert.pth"
target_path = os.path.join(target_dir, "lipsync_expert.pth")

print(f"Downloading lipsync_expert.pth to {target_path}...")
# -L follows redirects, -f fails on server errors
!curl -L -f {download_url} --output {target_path}

if os.path.exists(target_path) and os.path.getsize(target_path) > 1000000:
    print(f"Successfully downloaded: {target_path}")
    print(f"File size: {os.path.getsize(target_path) / (1024*1024):.2f} MB")
else:
    print("Download failed. The repository may be private or the link has changed.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1361  100  1361    0     0   3732      0 --:--:-- --:--:-- --:--:--  3728
100  188M  100  188M    0     0  20.6M      0  0:00:09  0:00:09 --:--:-- 21.8M
Successfully downloaded: /content/playground/pretrained_weights/lipsync_expert.pth
File size: 188.21 MB


In [ ]:
!unzip /content/s1.mpg_vcd.zip -d /content/playground/data/dataset/grid

Archive:  /content/s1.mpg_vcd.zip
   creating: /content/playground/data/dataset/grid/s1/
  inflating: /content/playground/data/dataset/grid/s1/swio1s.mpg  
  inflating: /content/playground/data/dataset/grid/s1/prii9a.mpg  
  inflating: /content/playground/data/dataset/grid/s1/sgwp9s.mpg  
  inflating: /content/playground/data/dataset/grid/s1/lwws5s.mpg  
  inflating: /content/playground/data/dataset/grid/s1/bbal8p.mpg  
  inflating: /content/playground/data/dataset/grid/s1/pwwrzp.mpg  
  inflating: /content/playground/data/dataset/grid/s1/pwwezn.mpg  
  inflating: /content/playground/data/dataset/grid/s1/sgivzn.mpg  
  inflating: /content/playground/data/dataset/grid/s1/swwi9s.mpg  
  inflating: /content/playground/data/dataset/grid/s1/lgwtzn.mpg  
  inflating: /content/playground/data/dataset/grid/s1/sgii2n.mpg  
  inflating: /content/playground/data/dataset/grid/s1/lwwm3a.mpg  
  inflating: /content/playground/data/dataset/grid/s1/pbio4n.mpg  
  inflating: /content/playground/data/da

### 下载 s1 的对齐文件 (Alignments) 和 音频文件 (Audio)
由于预处理脚本需要对齐文件和音频文件来提取特征，我们需要补充下载这些资源。

In [14]:
import os

# 定义目标目录
align_dir = "/content/playground/data/dataset/grid"
audio_dir = "/content/playground/data/dataset/grid"
os.makedirs(align_dir, exist_ok=True)
os.makedirs(audio_dir, exist_ok=True)

# 下载对齐文件
print("正在下载对齐文件 (s1.tar)...")
!curl -L https://spandh.dcs.shef.ac.uk/gridcorpus/s1/align/s1.tar --output /content/s1_align.tar
!tar -xf /content/s1_align.tar -C {align_dir}

# 下载音频文件
print("正在下载音频文件 (s1.tar)...")
!curl -L https://spandh.dcs.shef.ac.uk/gridcorpus/s1/audio/s1.tar --output /content/s1_audio.tar
!tar -xf /content/s1_audio.tar -C {audio_dir}

print("下载并解压完成。")

正在下载对齐文件 (s1.tar)...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1010k  100 1010k    0     0   604k      0  0:00:01  0:00:01 --:--:--  604k
正在下载音频文件 (s1.tar)...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 72.0M  100 72.0M    0     0  15.6M      0  0:00:04  0:00:04 --:--:-- 17.5M
下载并解压完成。


In [8]:
!pip install lpips

Looking in indexes: https://mirrors.tencent.com/pypi/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
!pip uninstall tensorflow tensorflow-metadata -y
!pip install "protobuf<5.0.0"
!pip install mediapipe==0.10.13


Looking in indexes: https://mirrors.tencent.com/pypi/simple/
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 1.1.0 requires websockets>=13.0, but you have websockets 12.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Looking in indexes: https://mirrors.tencent.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 9.2 MB/s  0:00:036m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 65.3 MB/s  0:00:00m0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4━━━━━ 

In [25]:
!apt-get update -y
!apt-get install ffmpeg -y

Hit:1 http://mirrors.cloud.tencent.com/ubuntu noble InRelease
Get:2 file:/var/cuda-repo-ubuntu2204-12-1-local  InRelease [1,572 B]           
Get:3 http://mirrors.cloud.tencent.com/ubuntu noble-updates InRelease [126 kB] 
Get:2 file:/var/cuda-repo-ubuntu2204-12-1-local  InRelease [1,572 B]           
Get:4 http://mirrors.cloud.tencent.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:6 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease [1,477 B]
Get:7 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]        
Get:8 https://nvidia.github.io/libnvidia-container/experimental/deb/amd64  InRelease [1,489 B]
Get:9 http://mirrors.cloud.tencent.com/ubuntu noble-updates/restricted amd64 Packages [3,824 kB]
Get:10 http://mirrors.cloud.tencent.com/ubuntu noble-updates/main amd64 Packages [2,412 kB]
Get:12 http://mirrors.cloud.tencent.com/ubuntu noble-updates/universe amd64 Packages 

In [ ]:
%cd playground
!python lip_sync/data_utils/preprocess_grid.py --limit 1000 --workers 1 --skip 3

/content/playground


/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
[*] Starting multiprocess preprocessing with 1 workers...
[*] Skip factor: 3 (compute every 3 frames)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
[*] 正在初始化 EchoMimic 模型 (已开启低内存优化)...
  0%|                                                  | 0/1000 [00:00<?, ?it/s]Some weights of the model checkpoint were not used when initializing UNet2DConditionModel: 
 ['down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_q.weight, down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k.weight, down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v.weight, down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.weight, down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_out.0.bias, down_blocks.0.atte

In [36]:
%cd /content/playground
!python lip_sync/train/train_text_to_feature.py

/content/playground
[*] Starting training on cuda
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|████████████████████████████████████████| 528M/528M [00:14<00:00, 37.6MB/s]
Loading model from

## 5. Inference (Test the pure text-driven flow)

Once the model is trained (`pretrained_weights/text_driven_model.pth`), you can generate video from pure text.

### 5.1 Download Pretrained Weights
The inference script expects EchoMimic weights and the SD VAE to be present in the `pretrained_weights` folder. Let's download them.

In [ ]:
!rm -rf /content/playground/third_party/EchoMimic/pretrained_weights/sd-image-variations-diffusers

In [37]:
%cd /content/playground
import sys
import os

# Re-running the generation script after the huggingface_hub fix
!python lip_sync/inference_pipeline.py

# View the generation result
from IPython.display import Video
if os.path.exists("output/text_driven_result/sample_video.mp4"):
    display(Video("output/text_driven_result/sample_video.mp4", embed=True))
else:
    print("Video file not found. Please check the logs above for errors.")

/content/playground
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(

[*] Starting text-driven pipeline...
[*] Input text: 'I am a text-driven talking head. No audio needed.'
[*] [Step 1] Processing text to visemes...
[*] Viseme sequence length: 44, Target frames: 352
[*] [Step 2&3] Mapping features + Generating video...
[*] 正在初始化 EchoMimic 模型 (已开启低内存优化)...
Some weights of the model checkpoint were not used when initializing UNet2DConditionModel: 
 ['down_blocks.0.attentions.0.transformer_blocks.0.attn